# Interactive wand tracking POC
Run on the Pi kernel. Hold a bright/high-contrast tip (phone flashlight, white LED, reflective tape) in front of the camera, ideally in a dim room. See `docs/WAND_POC.md`.

Defaults open no hardware. Set `RUN_CAMERA = True` on the Pi, or `DEMO_SYNTHETIC = True` to exercise tracking and the trace view with a simulated wand and no camera. Every cell is safe to re-run; **the last cell always releases everything** and can be run at any time.

## 1. Setup and parameters

In [ ]:
RUN_CAMERA = False  # True on the Pi to open the real camera
DEMO_SYNTHETIC = False  # True (with RUN_CAMERA False): simulated wand, no hardware

# Camera: measure what is actually stable (sections 5-6) before raising these.
CAMERA_SIZE = (640, 480)
CAMERA_FPS = 30  # controller accepts 1..60

# Tracking parameters (pixels in the camera frame)
THRESHOLD = 200
MIN_AREA = 20
MAX_AREA = 5000
MAX_TRAJECTORY_POINTS = 100
MAX_JUMP = 120
MAX_MISSED_FRAMES = 5
MIN_STROKE_POINTS = 8  # a lost wand with at least this many points ends a stroke
MOTION_THRESHOLD = None  # e.g. 25 to require the blob to be moving

# Display / diagnostics
DISPLAY_DELAY = 0.1  # seconds between overlay refreshes
TRACE_DELAY = 0.05  # seconds between trace refreshes: raise to slow the display down
TRACE_SECONDS = 20  # trace view stops itself after this long
REPLAY_DELAY = 0.04  # seconds per point when replaying a finished stroke
BENCH_SECONDS = 5
RUN_MODE_SWEEP = False  # section 6 closes the camera, tries several modes, then you re-run section 2
BENCH_MODES = [((320, 240), 30), ((640, 480), 30), ((640, 480), 60),
               ((1280, 720), 30), ((1280, 720), 60)]

# Optional arm demo (section 9). Moves real servos: off by default.
ENABLE_ARM_FOLLOW = False
STAND_AND_POWER_CUTOFF_READY = False  # robot on a stand, physical cutoff in reach
SERVO_CLEARANCE_VERIFIED = False  # pan/camera linkage can move freely
INVERT_X = False  # defaults match the legacy FPV.py tracker; flip if the arm turns away
INVERT_Y = False

import sys
import time
from pathlib import Path

# Make `import rasptank` work whether Jupyter was opened at the repo root or in notebooks/.
repo_root = next((p for p in (Path.cwd(), Path.cwd().parent)
                  if (p / 'rasptank' / '__init__.py').is_file()), None)
if repo_root is None:
    raise RuntimeError('Open this notebook from the repository or notebooks directory')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from rasptank import RaspTank, CameraController, MotorController, ServoController, RobotArm
from rasptank.wand import WandTracker, camera_source, path_length, bounding_box, direction_changes
from rasptank.wand_arm import WandArmFollower
from rasptank.wand_diagnostics import (LiveView, SyntheticWand, benchmark_camera,
                                       format_results, scene_report, sweep_modes)

# Everything below needs OpenCV/ipywidgets, which are only imported when a view is wanted.
# That keeps the default (hardware-free, dependency-free) run working anywhere.
LIVE = RUN_CAMERA or DEMO_SYNTHETIC
if LIVE:
    import cv2
    import ipywidgets as widgets
    from IPython.display import display

strokes = []  # finished strokes (N, 2 arrays), newest last

def remember_stroke(stroke):
    """Tracker callback: keep the 10 newest finished strokes for replay and gesture work."""
    strokes.append(stroke)
    del strokes[:-10]

def new_tracker(source=None, **overrides):
    """Build a WandTracker from the section 1 parameters; keyword overrides win.

    source: frame callable (see live_source) or None for process()-only use.
    This is the single place the notebook parameters reach the tracker, so
    every section (test frame, benchmark, live view) uses the same settings.
    """
    options = dict(threshold=THRESHOLD, min_area=MIN_AREA, max_area=MAX_AREA,
                   max_trajectory_points=MAX_TRAJECTORY_POINTS, max_jump=MAX_JUMP,
                   max_missed_frames=MAX_MISSED_FRAMES, motion_threshold=MOTION_THRESHOLD,
                   on_stroke=remember_stroke, min_stroke_points=MIN_STROKE_POINTS)
    options.update(overrides)
    return WandTracker(source, **options)

def release_all():
    """Stop views/trackers/follower, then close servos and camera. Returns (released, problems).

    Order matters: consumers (follower, views, tracker) stop before the things
    they read from (servos, camera) are closed. Each resource is handled
    independently so one failure cannot leave the camera locked, and every
    name is reset to None so the cell is safe to run repeatedly.
    """
    released, problems = [], []
    for name in ('follower', 'overlay_view', 'trace_view', 'tracker', 'servos', 'camera'):
        resource = globals().get(name)
        if resource is None:
            continue
        try:
            if name in ('servos', 'camera'):
                resource.close()
            else:
                resource.stop()
            released.append(name)
        except Exception as exc:
            problems.append(f'{name}: {exc}')
        globals()[name] = None
    return released, problems

## 2. Initialize RaspTank and camera
No motors are driven (dry-run motor controller); only the existing camera controller is started. Re-running releases the previous session first.

In [ ]:
# Free anything left from a previous run first, so re-running never leaks a locked camera.
_, problems = release_all()
if problems:
    print('cleanup problems:', problems)
camera = tank = None
if RUN_CAMERA:
    try:
        camera = CameraController(size=CAMERA_SIZE, fps=CAMERA_FPS)
        # Dry-run motors keep GPIO untouched; the camera is borrowed, so we close it ourselves.
        tank = RaspTank(motors=MotorController(use_gpio=False), camera=camera)
        tank.start_camera()
        print('Camera started')
    except Exception as exc:
        if camera is not None:
            camera.close()
            camera = None
        raise RuntimeError(f'Camera unavailable (in use by another process/kernel?): {exc}') from exc
elif DEMO_SYNTHETIC:
    print('Synthetic wand demo: no camera opened')
else:
    print('Camera disabled (RUN_CAMERA = False)')

def live_source():
    """Return a fresh frame source: the real camera, or a simulated wand when demoing.

    Anything returning an RGB array works here (USB camera, NoIR camera, video
    file): the tracker never sees the camera, only this callable.
    """
    return camera_source(camera) if RUN_CAMERA else SyntheticWand(CAMERA_SIZE, CAMERA_FPS)

## 3. Capture a test frame

In [ ]:
if RUN_CAMERA:
    # capture() blocks for the next frame; frame.rgb is packed RGB, frame.jpeg is for display.
    frame = camera.capture(timeout=2)
    print('size', (frame.width, frame.height), '| format: packed RGB, 3 bytes/pixel')
    display(widgets.Image(value=frame.jpeg, format='jpeg'))

## 4. Test image processing
Original, grayscale, threshold mask and detected candidates, plus a plain-language hint. Hold the wand tip in view and tune the parameters in section 1 until only the tip is a blob.

In [ ]:
if LIVE:
    # Separate tracker with the motion filter off so one still frame can show candidates.
    tester = new_tracker(motion_threshold=None)
    rgb = live_source()()
    gray, mask = tester.mask(rgb, update_motion=False)
    report = scene_report(rgb, tester)
    print('candidates (x, y, area):', report['candidates'])
    print(f"brightest pixel {report['brightest']}, {report['percent_over']:.2f}% above threshold")
    print('hint:', report['hint'])

    marked = rgb.copy()
    for x, y, area in report['candidates']:
        cv2.circle(marked, (int(x), int(y)), 10, (255, 0, 0), 2)

    def jpeg(image):
        """Encode an RGB or single-channel image as JPEG bytes for a widget."""
        if image.ndim == 2:
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        return cv2.imencode('.jpg', cv2.cvtColor(image, cv2.COLOR_RGB2BGR))[1].tobytes()

    display(widgets.HBox([widgets.Image(value=jpeg(img), format='jpeg', width=320)
                          for img in (rgb, gray, mask, marked)]))

## 5. Diagnostic: how fast and how late is this camera?
Runs the real tracker on every delivered frame for `BENCH_SECONDS` while you wave the wand. Read the numbers like this:
- **FPS** well below the requested rate: the camera mode, the JPEG + raw-RGB hand-off between processes, or processing is the limit.
- **dropped** frames: sequence numbers skipped, so fast strokes lose points.
- **latency** (capture start to arrival): how far the picture lags the real wand. Over ~150 ms feels laggy for arm following.
- **process** ms: tracker cost per frame; it must stay under 1000/FPS.
- **detected**: share of frames with a tip found; low means threshold/lighting problems, not speed.

In [ ]:
if RUN_CAMERA:
    # A fresh tracker per run: the benchmark feeds it every frame, so it must not share state.
    bench = benchmark_camera(camera, new_tracker(), BENCH_SECONDS)
    print(format_results([{'requested': (CAMERA_SIZE, CAMERA_FPS), **bench}]))
elif DEMO_SYNTHETIC:
    print('Benchmark needs the real camera (RUN_CAMERA = True)')

## 6. Diagnostic: compare camera modes (optional)
Set `RUN_MODE_SWEEP = True`. This **closes the current camera**, tries each `(size, fps)` in `BENCH_MODES` for `BENCH_SECONDS` each (reporting failures rather than stopping), and closes every camera it opens. Re-run section 2 afterwards. Requesting a mode does not guarantee the sensor delivers it: compare requested against measured FPS.

In [ ]:
if RUN_CAMERA and RUN_MODE_SWEEP:
    release_all()  # the camera is exclusive: free it for the sweep
    camera = tank = None
    print(format_results(sweep_modes(BENCH_MODES, new_tracker, BENCH_SECONDS)))
    print('Camera is closed; re-run section 2 to reopen it.')
else:
    print('Mode sweep disabled (needs RUN_CAMERA and RUN_MODE_SWEEP)')

## 7. Live tracking (camera view with trail)
`tracker.start()` runs tracking at camera speed in a background thread while the widget refreshes every `DISPLAY_DELAY` seconds. The **Stop view** button ends the display; `tracker.stop()` (or the last cell) ends tracking. Re-running replaces the previous session.

In [ ]:
if LIVE:
    # Re-run safety: stop the previous session's threads before replacing them.
    for name in ('overlay_view', 'trace_view', 'tracker'):
        if globals().get(name) is not None:
            globals()[name].stop()
    tracker = new_tracker(live_source())
    overlay_view = LiveView(tracker, mode='overlay', delay=DISPLAY_DELAY)  # display only
    tracker.start()  # tracking thread runs at camera speed, independent of the display

## 8. Diagnostic: white trace on black
Shows only the tracked tip and its trail, white on black, refreshed every `TRACE_DELAY` seconds (a deliberate slow-down so you can watch it; tracking itself is not slowed). It stops after `TRACE_SECONDS`. Needs section 7 running. Use it to judge trail smoothness, gaps and jitter without the camera picture getting in the way. A finished stroke is kept in `strokes`; the next cell replays it slowly.

In [ ]:
if LIVE:
    if globals().get('trace_view') is not None:
        trace_view.stop()
    trace_view = LiveView(tracker, mode='trace', delay=TRACE_DELAY, seconds=TRACE_SECONDS)

In [ ]:
# Replay the last finished stroke slowly, then print its numbers. A stroke ends when the
# wand is lost for more than MAX_MISSED_FRAMES frames (cover the tip or move it out of view).
if LIVE:
    if strokes:
        stroke = strokes[-1]
        trace_view.replay(stroke, tracker.frame_size, delay=REPLAY_DELAY)
        print('points', len(stroke), '| length', round(path_length(stroke), 1),
              '| bbox', bounding_box(stroke), '| turns', direction_changes(stroke))
    else:
        print('No finished stroke yet: draw something, then hide the tip briefly')

## 9. Optional: wand moves the arm
Set `ENABLE_ARM_FOLLOW`, `STAND_AND_POWER_CUTOFF_READY` and `SERVO_CLEARANCE_VERIFIED` in section 1 and run section 7 first. The follower starts **disarmed**; the toggle arms it. It only moves while the wand is tracked, takes small rate-limited steps, ignores a center dead band and never forces past a joint limit. Pan (arm left/right, which turns the camera) follows the wand horizontally and the camera tilt servo follows it vertically, the same mapping as the legacy `FPV.py` color tracker. Test on a stand first. Turning the toggle off, stopping tracking or running the last cell disarms it.

In [ ]:
if RUN_CAMERA and ENABLE_ARM_FOLLOW:
    if not (STAND_AND_POWER_CUTOFF_READY and SERVO_CLEARANCE_VERIFIED):
        raise RuntimeError('Complete the stand/cutoff and servo clearance checks first')
    if globals().get('follower') is not None:
        follower.stop()
    if globals().get('servos') is None:
        servos = ServoController()  # frequency only; nothing moves until armed
    # The follower only borrows tracker and arm; it stays disarmed until the toggle is switched on.
    follower = WandArmFollower(tracker, RobotArm(servos), invert_x=INVERT_X,
                               invert_y=INVERT_Y).start()
    arm_toggle = widgets.ToggleButton(description='Arm follower', value=False)
    arm_toggle.observe(lambda change, f=follower: setattr(f, 'armed', bool(change['new'])),
                       names='value')
    display(arm_toggle)
else:
    print('Arm demo disabled (needs RUN_CAMERA and ENABLE_ARM_FOLLOW)')

## 10. Stop and release everything (always run)
Also prints statistics of the trajectory a future gesture recognizer would use.

In [ ]:
if globals().get('tracker') is not None:
    trajectory = tracker.trajectory  # (N, 2) numpy array
    print('points', len(trajectory), '| length', round(path_length(trajectory), 1),
          '| bbox', bounding_box(trajectory), '| turns', direction_changes(trajectory))
released, problems = release_all()
print('Released:', released or 'nothing to release')
if problems:
    raise RuntimeError('Cleanup problems: ' + '; '.join(problems))